In [ ]:
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
room_categories_path = "/home/cgokmen/room_categories.txt"
with open(room_categories_path) as f:
    sem_to_id = {line.strip(): i + 1 for i, line in enumerate(f.readlines())}
id_to_sem = {v: k for k, v in sem_to_id.items()}

In [ ]:
# Check that all room types that show up on the list also show up on the image
import json

with open(
    "/scr/BEHAVIOR-1K/asset_pipeline/artifacts/pipeline/combined_room_object_list.json"
) as f:
    room_object_lists = json.load(f)["scenes"]
scene_room_instances = {
    scene: list(rooms.keys()) for scene, rooms in room_object_lists.items()
}
scene_room_types = {
    scene: {x.rsplit("_", 1)[0] for x in rooms}
    for scene, rooms in scene_room_instances.items()
}
assert all(room in sem_to_id for rooms in scene_room_types.values() for room in rooms)

In [ ]:
scene_room_types

In [ ]:
import fs.zipfs
import fs.path

maps_fs = fs.zipfs.ZipFS(
    "/scr/BEHAVIOR-1K/asset_pipeline/artifacts/parallels/scenes_json.zip"
)
valid_scenes = set(scene_room_types.keys())

In [ ]:
import imagesize
import io

scene_map_sizes = {
    scene: [
        x * 0.01
        for x in imagesize.get(
            io.BytesIO(
                maps_fs.readbytes(
                    fs.path.join("scenes", scene, "layout", "floor_semseg_0.png")
                )
            )
        )
    ]
    for scene in valid_scenes
}
sorted(scene_map_sizes.items(), key=lambda x: -x[1][0] * x[1][1])

In [ ]:
SKIP_SCENES = {"hall_arch_wood"}
scene_maps = {
    scene: np.array(
        Image.open(
            maps_fs.open(
                fs.path.join("scenes", scene, "layout", "floor_semseg_0.png"), "rb"
            )
        )
    ).astype(int)
    for scene in valid_scenes
    if scene not in SKIP_SCENES
}

In [ ]:
valid_scenes

In [ ]:
room_ids_in_map = {
    scene: set(np.unique(img)) - {0} for scene, img in scene_maps.items()
}
room_types_in_map = {
    scene: {id_to_sem[x] for x in room_ids}
    for scene, room_ids in room_ids_in_map.items()
}

In [ ]:
for scene in valid_scenes - SKIP_SCENES:
    expected = scene_room_types[scene]
    found = room_types_in_map[scene]
    if expected != found:
        print("Problem in", scene)
        print("Expected", expected)
        print("Found", found)
        print("Missing", expected - found)
        print("Unexpected", found - expected)
        print()

In [ ]:
import matplotlib.pyplot as plt

trav_maps = {
    scene: np.array(
        Image.open(
            maps_fs.open(
                fs.path.join("scenes", scene, "layout", "floor_trav_0.png"), "rb"
            )
        )
    ).astype(int)
    for scene in valid_scenes
    if scene not in SKIP_SCENES
}
for scene, map_img in trav_maps.items():
    plt.imshow(map_img)
    plt.title(scene)
    plt.show()